## 1) Organise the dataset

In [1]:
# 1.1) Robust project_root + imports
import sys, json
from pathlib import Path

# Ensure project root is on PYTHONPATH
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.data.importData import main as import_main

RAW_ROOT  = project_root / "data" / "raw"
PROC_ROOT = project_root / "data" / "processed"

In [2]:
# 1.2) Verify-or-refill: if processed exists but is incomplete, run import_main()
def _subset_counts(u_dir: Path) -> dict:
    gf = list((u_dir / "GlobalFeatures").glob("*.mat"))
    lf = list((u_dir / "LocalFunctions").glob("*.mat"))
    return {"gf": len(gf), "lf": len(lf)}

EXPECTED_PER_SUBSET = 28  # 4 sessions × 7 attempts

user_dirs = [d for d in PROC_ROOT.glob("u*") if d.is_dir()]

needs_import = False
if user_dirs:
    # Check a few users and also fail fast if any user is incomplete
    bad = []
    for u in user_dirs:
        c = _subset_counts(u)
        if (c["gf"] != EXPECTED_PER_SUBSET) or (c["lf"] != EXPECTED_PER_SUBSET):
            bad.append((u.name, c))
    if bad:
        print(f"⚠️ Found {len(bad)} users with incomplete processed data. Re-running import_main() to refill.")
        needs_import = True
    else:
        print(f"✅ Found {len(user_dirs)} complete user folders under {PROC_ROOT}.")
else:
    print("⏳ No per-user folders found under processed.")

if (not user_dirs) or needs_import:
    import_main(raw_root=RAW_ROOT, proc_root=PROC_ROOT)
    user_dirs = [d for d in PROC_ROOT.glob("u*") if d.is_dir()]
    print(f"✅ After import, {len(user_dirs)} user folders under {PROC_ROOT}")

# 1.3) Sanity-check layout for first few users
for u in sorted(user_dirs)[:3]:
    c = _subset_counts(u)
    print(f"{u.name}: GlobalFeatures={c['gf']} files, LocalFunctions={c['lf']} files")

✅ Found 400 complete user folders under c:\Users\mattt\Skripsie\Projects\DTW-project\data\processed.
u1001: GlobalFeatures=28 files, LocalFunctions=28 files
u1002: GlobalFeatures=28 files, LocalFunctions=28 files
u1003: GlobalFeatures=28 files, LocalFunctions=28 files


## 2) Build the catalog index + deterministic splits

In [3]:
from pathlib import Path
import pandas as pd
import numpy as np
from src.io.load_biosecurid import build_catalog, make_splits

CATALOG_PATH = project_root / "data" / "catalog" / "biosecurid_catalog.parquet"
SPLITS_PATH  = project_root / "data" / "splits"  / "user_splits.json"
CATALOG_PATH.parent.mkdir(parents=True, exist_ok=True)
SPLITS_PATH.parent.mkdir(parents=True, exist_ok=True)

# Build or load (rebuild if empty/missing or missing new columns)
required_cols = {"sig_name", "has_gf", "n_timesteps", "feature_dim"}
if CATALOG_PATH.exists():
    df_catalog = pd.read_parquet(CATALOG_PATH, engine="pyarrow")
    if df_catalog.empty or not required_cols.issubset(df_catalog.columns):
        print("⚠️ Catalog is empty or missing required columns — rebuilding…")
        df_catalog = build_catalog(PROC_ROOT, CATALOG_PATH)
    else:
        print("✅ Loaded existing catalog.")
else:
    print("⏳ Building new catalog…")
    df_catalog = build_catalog(PROC_ROOT, CATALOG_PATH)

print(f"Rows in catalog: {len(df_catalog)}")

# Preview + info
try:
    display(df_catalog.head(8))
except NameError:
    print(df_catalog.head(8))
print(df_catalog.dtypes)

# Persist deterministic user splits + add 'split' column into the catalog itself
dev_users, test_users = make_splits(df_catalog, SPLITS_PATH)
df_catalog["split"] = np.where(df_catalog["user"].astype(int).isin(dev_users), "dev", "test")

# Add aliases to mirror pipeline naming
df_catalog["n_timesteps"] = df_catalog["n_rows_lf"]
df_catalog["feature_dim"]  = df_catalog["n_cols_lf"]

# Save the enriched catalog (with 'split', 'sig_name', 'has_gf')
df_catalog.to_parquet(CATALOG_PATH, index=False, engine="pyarrow")
print(f"✅ Wrote enriched catalog to {CATALOG_PATH}")
print(f"✅ Wrote splits to {SPLITS_PATH}")
print(f"dev_users: {len(dev_users)} | test_users: {len(test_users)}")

⚠️ Catalog is empty or missing required columns — rebuilding…
Rows in catalog: 11200


,path_gf,path_lf,user,session,attempt,label,n_rows_lf,n_cols_lf,sig_name,has_gf
0,data\processed\u1001\GlobalFeatures\u1001s0001...,data\processed\u1001\LocalFunctions\u1001s0001...,1001,1,1,genuine,202,9,u1001s0001_sg0001,True
1,data\processed\u1001\GlobalFeatures\u1001s0001...,data\processed\u1001\LocalFunctions\u1001s0001...,1001,1,2,genuine,201,9,u1001s0001_sg0002,True
2,data\processed\u1001\GlobalFeatures\u1001s0001...,data\processed\u1001\LocalFunctions\u1001s0001...,1001,1,3,skilled,686,9,u1001s0001_sg0003,True
3,data\processed\u1001\GlobalFeatures\u1001s0001...,data\processed\u1001\LocalFunctions\u1001s0001...,1001,1,4,skilled,778,9,u1001s0001_sg0004,True
4,data\processed\u1001\GlobalFeatures\u1001s0001...,data\processed\u1001\LocalFunctions\u1001s0001...,1001,1,5,skilled,578,9,u1001s0001_sg0005,True
5,data\processed\u1001\GlobalFeatures\u1001s0001...,data\processed\u1001\LocalFunctions\u1001s0001...,1001,1,6,genuine,192,9,u1001s0001_sg0006,True
6,data\processed\u1001\GlobalFeatures\u1001s0001...,data\processed\u1001\LocalFunctions\u1001s0001...,1001,1,7,genuine,195,9,u1001s0001_sg0007,True
7,data\processed\u1001\GlobalFeatures\u1001s0002...,data\processed\u1001\LocalFunctions\u1001s0002...,1001,2,1,genuine,199,9,u1001s0002_sg0001,True


path_gf      object
path_lf      object
user          int64
session       int64
attempt       int64
label        object
n_rows_lf     int64
n_cols_lf     int64
sig_name     object
has_gf         bool
dtype: object
✅ Wrote enriched catalog to c:\Users\mattt\Skripsie\Projects\DTW-project\data\catalog\biosecurid_catalog.parquet
✅ Wrote splits to c:\Users\mattt\Skripsie\Projects\DTW-project\data\splits\user_splits.json
dev_users: 300 | test_users: 100


In [4]:
# 2.1) Integrity checks (fail fast)

# Exactly 9 columns in LocalFunctions everywhere
bad_shapes = df_catalog[df_catalog["n_cols_lf"] != 9]
assert bad_shapes.empty, f"Found {len(bad_shapes)} files with n_cols_lf != 9"

# Dataset should have exactly 400 users
n_users = df_catalog["user"].nunique()
assert n_users == 400, f"Expected 400 users; got {n_users}"

# Each user should have 4 sessions × 7 attempts = 28 attempts (LocalFunctions)
per_user_counts = df_catalog.groupby("user").size()
bad_users = per_user_counts[per_user_counts != 28]
if not bad_users.empty:
    try:
        display(bad_users.head())
    except NameError:
        print(bad_users.head())
    raise AssertionError(f"Users with unexpected attempt counts: {len(bad_users)}")

# Label mix per session should be 4 genuine, 3 skilled
mix = (df_catalog
       .groupby(["user", "session", "label"])
       .size()
       .unstack(fill_value=0)
       .rename(columns={"genuine": "n_genuine", "skilled": "n_skilled"}))

bad_sessions = mix[(mix["n_genuine"] != 4) | (mix["n_skilled"] != 3)]
if not bad_sessions.empty:
    try:
        display(bad_sessions.head(10))
    except NameError:
        print(bad_sessions.head(10))
    raise AssertionError(f"Sessions with wrong label counts: {len(bad_sessions)}")

# New: no duplicate (user, session, attempt)
dupes = df_catalog.duplicated(["user", "session", "attempt"], keep=False)
assert not dupes.any(), f"Duplicate attempt rows: {dupes.sum()}"

# New: GF must exist everywhere (unless you explicitly accept missing GF)
assert df_catalog["has_gf"].all(), "Missing GlobalFeatures for some attempts."

# New: no NaNs and no zero/negative lengths
essential_cols = ["user", "session", "attempt", "label", "n_rows_lf", "n_cols_lf", "path_lf"]
null_counts = df_catalog[essential_cols].isnull().sum()
if int(null_counts.sum()) > 0:
    print(null_counts[null_counts > 0])
    raise AssertionError("Nulls found in essential catalog columns.")
assert (df_catalog["n_rows_lf"] > 0).all(), "Found attempts with zero/negative n_rows_lf."

# New: flag (don't fail) extreme length outliers for inspection
q1, q3 = np.percentile(df_catalog["n_rows_lf"], [25, 75])
iqr = q3 - q1
lower, upper = max(1, q1 - 3 * iqr), q3 + 3 * iqr
outliers = df_catalog[(df_catalog["n_rows_lf"] < lower) | (df_catalog["n_rows_lf"] > upper)]
if not outliers.empty:
    print(f"⚠️ Length outliers detected: {len(outliers)} rows (n_rows_lf not in [{lower:.1f}, {upper:.1f}]).")
    try:
        display(outliers[["user","session","attempt","n_rows_lf"]].head(10))
    except NameError:
        print(outliers[["user","session","attempt","n_rows_lf"]].head(10).to_string(index=False))

print("✅ Catalog integrity checks passed.")

⚠️ Length outliers detected: 234 rows (n_rows_lf not in [1.0, 2642.0]).


,user,session,attempt,n_rows_lf
142,1006,1,3,2662
150,1006,2,4,2895
151,1006,2,5,3229
158,1006,3,5,2969
205,1008,2,3,2861
212,1008,3,3,3025
683,1025,2,5,3416
739,1027,2,5,3337
760,1028,1,5,2733
1047,1038,2,5,3681


✅ Catalog integrity checks passed.


In [5]:
# 2.2) Write a small catalog meta JSON for quick inspection
meta = {
    "rows": int(len(df_catalog)),
    "n_users": int(n_users),
    "counts": {
        "genuine": int((df_catalog["label"] == "genuine").sum()),
        "skilled": int((df_catalog["label"] == "skilled").sum()),
    },
    "n_rows_lf_stats": {
        "min": int(df_catalog["n_rows_lf"].min()),
        "q1": float(q1),
        "median": float(df_catalog["n_rows_lf"].median()),
        "q3": float(q3),
        "max": int(df_catalog["n_rows_lf"].max()),
        "iqr": float(iqr),
        "outliers": int(len(outliers)),
    },
    "has_gf_missing": int((~df_catalog["has_gf"]).sum()),
}
meta_path = CATALOG_PATH.with_suffix(".meta.json")
meta_path.write_text(json.dumps(meta, indent=2))
print(f"📝 Wrote catalog meta → {meta_path}")

📝 Wrote catalog meta → c:\Users\mattt\Skripsie\Projects\DTW-project\data\catalog\biosecurid_catalog.meta.json
